In [ ]:
%load_ext jupyter_black

In [ ]:
# SPLITS = 'random',
# SPLITS = 'scaffold',
# SPLITS = 'tanimoto',
SPLITS = "temporal"

RUN_HYPEROPT = False
RUN_VALID = False
RUN_TEST = False

# Index


<div class="alert alert-block alert-info" style="margin-top: 20px">

<font size = 4>
1. <a href="#Imports">Imports</a><br>
2. <a href="#Data-preparation">Data preparation</a><br>
3. <a href="#Hyperopt">Hyperopt</a><br>
4. <a href="#Model-validation">Model validation</a><br>
5. <a href="#Test-set-prediction">Test set prediction</a><br>
6. <a href="#Domain-of-applicability">Domain of applicability</a><br>
7. <a href="#UMAP">UMAP</a><br>

</font>
</div>

# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import scipy.stats as ss
import seaborn as sns

from catboost import CatBoostClassifier
from itertools import combinations
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    matthews_corrcoef,
    recall_score,
    classification_report,
)
from sklearn.model_selection import ParameterGrid
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from umap import UMAP

sns.set(style="white", color_codes=True, palette="colorblind", font_scale=1)
tqdm.pandas()

# Data preparation

In [ ]:
dilirank = pd.read_excel(
    "../data/Supporting Information 2.xlsx",
    sheet_name="Table S1 - DILIrank",
)

dilirank = dilirank.drop(
    columns=[
        "SMILES",
        "InChIKey",
        "Approval Year",
        "DILIrank Concern",
        "Ionic Character",
        "Stereoisomerism",
        "BDDCS",
        "BSEP",
        "GSH",
    ]
)

dilirank.rename(
    columns={
        "DILIrank Class": "y",
        "Normalized ph7.4 SMILES": "smiles",
        "Normalized ph7.4 InChIKey": "inchikey",
    },
    inplace=True,
)

In [ ]:
mfpgen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=1024,
    includeChirality=False,
    useBondTypes=True,
)

dilirank["morgan"] = dilirank["smiles"].apply(
    lambda x: list(mfpgen.GetFingerprint(Chem.MolFromSmiles(x)))
)

In [ ]:
apgen = rdFingerprintGenerator.GetAtomPairGenerator(
    fpSize=1024,
    includeChirality=False,
)

dilirank["atom_pair"] = dilirank["smiles"].apply(
    lambda x: list(apgen.GetFingerprint(Chem.MolFromSmiles(x)))
)

In [ ]:
ttgen = rdFingerprintGenerator.GetTopologicalTorsionGenerator(
    fpSize=1024,
    includeChirality=False,
)

dilirank["topological_torsion"] = dilirank["smiles"].apply(
    lambda x: list(ttgen.GetFingerprint(Chem.MolFromSmiles(x)))
)

In [ ]:
splits_col = f"tier0_{SPLITS}_splits"

df_training = dilirank[~dilirank[splits_col].isin(["valid", "test"])].copy()
df_validation = dilirank[dilirank[splits_col] == "valid"].copy()
df_test = dilirank[dilirank[splits_col] == "test"].copy()

In [ ]:
volsurf_cols = [
    "V",
    "S",
    "R",
    "G",
    "d",
    "W1",
    "W2",
    "W3",
    "W4",
    "W5",
    "W6",
    "W7",
    "W8",
    "D1",
    "D2",
    "D3",
    "D4",
    "D5",
    "D6",
    "D7",
    "D8",
    "WO1",
    "WO2",
    "WO3",
    "WO4",
    "WO5",
    "WO6",
    "WN1",
    "WN2",
    "WN3",
    "WN4",
    "WN5",
    "WN6",
    "IW1",
    "IW2",
    "IW3",
    "IW4",
    "CW1",
    "CW2",
    "CW3",
    "CW4",
    "CW5",
    "CW6",
    "CW7",
    "CW8",
    "ID1",
    "ID2",
    "ID3",
    "ID4",
    "CD1",
    "CD2",
    "CD3",
    "CD4",
    "CD5",
    "CD6",
    "CD7",
    "CD8",
    "HL1",
    "HL2",
    "A",
    "CP",
    "POL",
    "MW",
    "FLEX",
    "FLEX_RB",
    "NCC",
    "DIFF",
    "SE",
    "SE0",
    "SE1",
    "LOGP n-Oct",
    "LOGP c-Hex",
    "PSA",
    "HSA",
    "PSAR",
    "PHSAR",
    "LgD5",
    "LgD6",
    "LgD7",
    "LgD7.5",
    "LgD8",
    "LgD9",
    "LgD10",
    "AUS7.4",
    "%FU4",
    "%FU5",
    "%FU6",
    "%FU7",
    "%FU8",
    "%FU9",
    "%FU10",
    "SOLY",
    "LgS3",
    "LgS4",
    "LgS5",
    "LgS6",
    "LgS7",
    "LgS7.5",
    "LgS8",
    "LgS9",
    "LgS10",
    "LgS11",
    "MpKaA",
    "mpKaB",
    "PB",
    "VD",
    "CACO2",
    "SKIN",
    "LgBB",
    "MetStab",
    "HTSflag",
    "C1",
    "C2",
    "C3",
    "C4",
    "C5",
    "P1",
    "P2",
    "P3",
    "P4",
    "P5",
    "EMDIF",
    "EMDIS",
]

In [ ]:
scaler = StandardScaler()
df_training["volsurf"] = scaler.fit_transform(df_training[volsurf_cols]).tolist()

# Hyperopt

In [ ]:
catboost_classifier = {
    "auto_class_weights": ["Balanced"],
    "random_state": [0],
    "verbose": [False],
    # "max_depth": list(np.arange(4, 11, 1)),
    "n_estimators": list(np.arange(400, 1100, 100)),
}

In [ ]:
x_features = [
    # "atom_pair",
    "morgan",
    # "topological_torsion",
    # "volsurf",
]

In [ ]:
if RUN_HYPEROPT:
    for feature in x_features:
        df_xy = df_training.copy()
        results = []
        print(
            "************************************************\nRunning catboost_classifier with %s\n************************************************"
            % (feature)
        )
        parameter_grid = list(ParameterGrid(catboost_classifier))
        G = []
        for p, param in enumerate(parameter_grid):
            for f, fold in enumerate(sorted(df_xy[splits_col].unique())):
                param_copy = param.copy()
                param_copy["X_tr"] = df_xy.loc[df_xy[splits_col] != fold][
                    feature
                ].tolist()
                param_copy["X_te"] = df_xy.loc[df_xy[splits_col] == fold][
                    feature
                ].tolist()
                param_copy["y_tr"] = df_xy.loc[df_xy[splits_col] != fold]["y"].tolist()
                param_copy["y_te"] = df_xy.loc[df_xy[splits_col] == fold]["y"].tolist()
                param_copy.update({"k_fold": f, "model_id": p})
                G.append(param_copy)
        for config in tqdm(G):
            X_tr, X_te = config["X_tr"], config["X_te"]
            y_tr, y_te = config["y_tr"], config["y_te"]
            model_params = {
                k: config[k]
                for k in config.keys()
                if k not in ["X_tr", "X_te", "y_tr", "y_te", "model_id", "k_fold"]
            }
            model = CatBoostClassifier(**model_params)
            model.fit(X=X_tr, y=y_tr)
            y_pred = model.predict(X_te)
            recall = recall_score(y_te, y_pred, average=None, labels=[1, 2, 3])
            dict_results = {
                "x_features": feature,
                "model_id": config["model_id"],
                "k_fold": config["k_fold"],
                "balanced_accuracy": balanced_accuracy_score(y_te, y_pred),
                "recall_class1": recall[0],
                "recall_class2": recall[1],
                "recall_class3": recall[2],
                "matthews_corrcoef": matthews_corrcoef(y_te, y_pred),
            }
            dict_results.update(model_params)
            df_results = pd.DataFrame(dict_results, index=[0])
            results.append(df_results)
        pd.concat(results, ignore_index=True).to_csv(
            "../results/tier0_%s_hyperopt_%s.csv" % (SPLITS, feature), index=False
        )

In [ ]:
F = []
for f in os.listdir("../results"):
    if f.startswith(f"tier0_{SPLITS}_hyperopt_"):
        df_csv = pd.read_csv(os.path.join("../results", f))
        F.append(df_csv)
df_model_data = pd.concat(F)

In [ ]:
df_model_data_mean = (
    df_model_data.drop(columns=["k_fold"])
    .groupby(
        by=[
            c
            for c in df_model_data.columns
            if c
            not in [
                "balanced_accuracy",
                "recall_class1",
                "recall_class2",
                "recall_class3",
                "matthews_corrcoef",
                "k_fold",
            ]
        ],
        dropna=False,
    )
    .mean()
    .reset_index()
)

In [ ]:
mean_order = (
    df_model_data_mean.groupby("x_features")["matthews_corrcoef"]
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.boxplot(
    data=df_model_data_mean,
    y="matthews_corrcoef",
    x="x_features",
    order=mean_order,
)

In [ ]:
T = []
norm_pval = 0.05
for x in df_model_data_mean["x_features"].unique():
    df_x = df_model_data_mean[df_model_data_mean["x_features"] == x]
    x_ba = df_x["matthews_corrcoef"].values
    x_ba_shapiro_pvalue = ss.shapiro(x_ba).pvalue
    value = 0.05
    if x_ba_shapiro_pvalue < 0.05:
        operator = "<"
        norm_pval = x_ba_shapiro_pvalue
        if x_ba_shapiro_pvalue < 0.01:
            value = 0.01
            if x_ba_shapiro_pvalue < 0.001:
                value = 0.001
    else:
        operator = "≥"
    T.append(
        pd.DataFrame(
            {
                "X": [x],
                "test": ["shapiro_wilk"],
                "operator": [operator],
                "pvalue": [value],
            }
        )
    )
pd.concat(T, ignore_index=True)

In [ ]:
df_pivoted = df_model_data_mean.pivot_table(
    index=["model_id"],
    columns="x_features",
    values="matthews_corrcoef",
)

In [ ]:
pairs = list(combinations(mean_order, 2))
ordered_pairs = []

current_pair = pairs[0]
ordered_pairs.append(current_pair)

for _ in range(1, len(pairs)):
    next_pair = next(
        (
            pair
            for pair in pairs
            if pair[0] == current_pair[1] and pair not in ordered_pairs
        ),
        None,
    )
    if next_pair:
        ordered_pairs.append(next_pair)
        current_pair = next_pair
    else:
        break

In [ ]:
T = []
i = 0
for x1, x2 in ordered_pairs:
    if norm_pval < 0.05:
        test_name = "wilcoxon"
        posthoc_pvalue = ss.wilcoxon(
            x=df_pivoted[x1].values,
            y=df_pivoted[x2].values,
        ).pvalue
    else:
        test_name = "t-test"
        posthoc_pvalue = ss.ttest_rel(
            a=df_pivoted[x1].values,
            b=df_pivoted[x2].values,
        ).pvalue
    value = 0.05
    if posthoc_pvalue < 0.05:
        operator = "<"
        i = 1
        if posthoc_pvalue < 0.01:
            value = 0.01
            if posthoc_pvalue < 0.001:
                value = 0.001
    else:
        operator = "≥"
    T.append(
        pd.DataFrame(
            {
                "x1": [x1],
                "x2": [x2],
                "test": [test_name],
                "operator": [operator],
                "pvalue": [value],
            }
        )
    )
    i += 1
df_t = pd.concat(T, ignore_index=True)
df_t

In [ ]:
X = []
for index, row in df_t.iterrows():
    if row["pvalue"] >= 0.05:
        X.append(row["x1"])
        X.append(row["x2"])
    else:
        X.append(row["x1"])
        break
selected_x = list(dict.fromkeys(X))
selected_x

# Model validation

In [ ]:
filtered_df = df_model_data_mean[df_model_data_mean["x_features"].isin(selected_x)]

In [ ]:
df_validation["volsurf"] = scaler.transform(df_validation[volsurf_cols]).tolist()

In [ ]:
if RUN_VALID:
    R = []
    for index, row in tqdm(filtered_df.iterrows()):
        params = row[
            [
                "auto_class_weights",
                "max_depth",
                "n_estimators",
                "random_state",
                "verbose",
            ]
        ]
        model_params = {
            params.index[x]: params.values[x] for x in range(params.shape[0])
        }
        X_tr, X_te = (
            df_training[row["x_features"]].tolist(),
            df_validation[row["x_features"]].tolist(),
        )
        y_tr, y_te = df_training["y"].tolist(), df_validation["y"].tolist()
        model = CatBoostClassifier(**model_params)
        model.fit(X=X_tr, y=y_tr)
        y_pred = model.predict(X_te)
        recall = recall_score(y_te, y_pred, average=None, labels=[1, 2, 3])
        dict_results = {
            "model_id": row["model_id"],
            "x_features": row["x_features"],
            "balanced_accuracy_val": balanced_accuracy_score(y_te, y_pred),
            "recall_class1_val": recall[0],
            "recall_class2_val": recall[1],
            "recall_class3_val": recall[2],
            "matthews_corrcoef_val": matthews_corrcoef(y_te, y_pred),
        }
        dict_results.update(model_params)
        R.append(pd.DataFrame(dict_results, index=[0]))
    df_results_val = pd.concat(R)
    df_results_val.to_csv(f"../results/tier0_{SPLITS}_valid_results.csv", index=False)
else:
    df_results_val = pd.read_csv(f"../results/tier0_{SPLITS}_valid_results.csv")

In [ ]:
df_merged = pd.merge(
    left=filtered_df.rename(
        columns={
            "balanced_accuracy": "balanced_accuracy_tra",
            "recall_class1": "recall_class1_tra",
            "recall_class2": "recall_class2_tra",
            "recall_class3": "recall_class3_tra",
            "matthews_corrcoef": "matthews_corrcoef_tra",
        }
    ),
    right=df_results_val.drop(
        columns=[
            "auto_class_weights",
            "max_depth",
            "n_estimators",
            "random_state",
            "verbose",
        ]
    ),
    on=["x_features", "model_id"],
)

In [ ]:
sns.scatterplot(
    df_merged, x="matthews_corrcoef_tra", y="matthews_corrcoef_val", hue="x_features"
)

In [ ]:
df_merged["sum_col1_col2"] = (
    df_merged["matthews_corrcoef_tra"] * df_merged["matthews_corrcoef_val"]
)

result = df_merged.loc[df_merged["sum_col1_col2"].idxmax()]

result["x_features"]

# Test set prediction

In [ ]:
df_model = df_merged[
    (df_merged["model_id"] == result["model_id"])
    & (df_merged["x_features"] == result["x_features"])
]

In [ ]:
df_training_val = dilirank[dilirank[splits_col] != "test"].copy()

In [ ]:
df_training_val["volsurf"] = scaler.fit_transform(df_training_val[volsurf_cols]).tolist()
df_test["volsurf"] = scaler.transform(df_test[volsurf_cols]).tolist()

In [ ]:
if RUN_TEST:
    params = df_model[
        ["auto_class_weights", "max_depth", "n_estimators", "random_state", "verbose"]
    ]
    X_tr, X_te = (
        df_training_val[result["x_features"]].tolist(),
        df_test[result["x_features"]].tolist(),
    )
    y_tr, y_te = df_training_val["y"].tolist(), df_test["y"].tolist()
    model = CatBoostClassifier(**params.to_dict(orient="records")[0])
    model.fit(X=X_tr, y=y_tr)
    y_pred = model.predict(X_te)
    df_test["y_pred"] = y_pred
    recall = recall_score(y_te, y_pred, average=None, labels=[1, 2, 3])
    dict_results = {
        "x_features": result["x_features"],
        "balanced_accuracy_tes": balanced_accuracy_score(y_te, y_pred),
        "recall_class1_tes": recall[0],
        "recall_class2_tes": recall[1],
        "recall_class3_tes": recall[2],
        "matthews_corrcoef_tes": matthews_corrcoef(y_te, y_pred),
    }
    dict_results.update(params.to_dict(orient="records")[0])
    df_results_tes = pd.DataFrame(dict_results, index=[0])
    df_results_tes.to_csv(f"../results/tier0_{SPLITS}_test_results.csv", index=False)
    df_test[["inchikey", "y_pred"]].to_csv(
        f"../results/tier0_{SPLITS}_test_predictions.csv", index=False
    )
else:
    df_results_tes = pd.read_csv(f"../results/tier0_{SPLITS}_test_results.csv")
    df_test = pd.merge(
        df_test,
        pd.read_csv(f"../results/tier0_{SPLITS}_test_predictions.csv"),
        on="inchikey",
    )

In [ ]:
print(
    classification_report(df_test["y"].tolist(), df_test["y_pred"].tolist(), digits=2)
)

In [ ]:
cm = confusion_matrix(
    y_true=df_test["y"].tolist(),
    y_pred=df_test["y_pred"].tolist(),
    labels=[1, 2, 3],
)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[1, 2, 3])
disp.plot()

# Domain of applicability

In [ ]:
knn = NearestNeighbors(n_neighbors=5, n_jobs=-1, p=2)
X = df_training_val[volsurf_cols].values.tolist()
# X = df_training_val["morgan"].tolist()
knn.fit(X)
con_nbrs = knn.kneighbors(X, return_distance=True, n_neighbors=6)
D = np.mean([np.mean(n[1:]) for n in con_nbrs[0]])
std = np.std([np.mean(n[1:]) for n in con_nbrs[0]])
doa_threshold = D + std * 0.5

In [ ]:
# samples = df_test["morgan"].tolist()
samples = df_test[volsurf_cols].values.tolist()
samples_nbrs = knn.kneighbors(samples, return_distance=True, n_neighbors=5)
df_test["mean_distance"] = [np.mean(n) for n in samples_nbrs[0]]
df_test.loc[
    df_test["mean_distance"] <= doa_threshold,
    "in_domain",
] = "yes"
df_test.loc[
    df_test["mean_distance"] > doa_threshold,
    "in_domain",
] = "no"

In [ ]:
df_test["in_domain"].value_counts()

In [ ]:
df_results_tes

In [ ]:
y_e = df_test[df_test["in_domain"] == "yes"]["y"]
y_p = df_test[df_test["in_domain"] == "yes"]["y_pred"]
recall = recall_score(y_e, y_p, average=None, labels=[1, 2, 3])
dict_results = {
    "balanced_accuracy": balanced_accuracy_score(y_e, y_p),
    "recall_class1": recall[0],
    "recall_class2": recall[1],
    "recall_class3": recall[2],
    "matthews_corrcoef": matthews_corrcoef(y_e, y_p),
}
pd.DataFrame(dict_results, index=[0]).round(2)

In [ ]:
round(len(df_test[df_test["in_domain"] == "no"]) / len(df_test), 2)

In [ ]:
cm = confusion_matrix(
    y_true=y_e,
    y_pred=y_p,
    labels=[1, 2, 3],
)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[1, 2, 3])
disp.plot()

# UMAP

In [ ]:
morgan_umap = UMAP(
    metric="manhattan",
    random_state=0,
)

df_umap = dilirank.copy()
X_umap = morgan_umap.fit_transform(df_umap["morgan"].tolist())

umap_cols = ["morgan_umap%s" % (i + 1) for i in range(len(X_umap[0]))]

df_umap[umap_cols] = pd.DataFrame(
    {"UMAP%s" % i: X_umap[:, i] for i in range(len(X_umap[0]))},
    index=df_umap.index,
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(
    data=df_umap[df_umap[splits_col] != "test"],
    x="morgan_umap1",
    y="morgan_umap2",
    ax=ax,
)
sns.scatterplot(
    data=df_umap[df_umap[splits_col] == "test"],
    x="morgan_umap1",
    y="morgan_umap2",
    ax=ax,
)

In [ ]:
X_fit = df_umap[df_umap[splits_col] != "test"][umap_cols]
X_predict = df_umap[df_umap[splits_col] == "test"][umap_cols]

df_dist = pd.DataFrame(
    {
        "X": umap_cols,
        "distances": list(
            map(
                lambda i: ss.ks_2samp(X_fit.iloc[:, i], X_predict.iloc[:, i]).pvalue,
                range(X_fit.shape[1]),
            )
        ),
    }
)

df_dist["distances"].mean().round(2)

### <span style="color:black"> <b><center> End of the notebook </span></b></center> <a id='test_configuration_file'></a>
***
!-- [Back to Index](#Index) -->